In [31]:
import os
import ast
import pickle
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz,utils
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dataclasses import dataclass, field
from typing import Optional

In [3]:
df = pd.read_csv(r'D:\Download\doc\DATASET\datset_kdrama\final\movies_merged.csv')

In [4]:
print(f"Jumlah baris: {df.shape[0]}, Jumlah kolom: {df.shape[1]}")
df.head(2)

Jumlah baris: 82830, Jumlah kolom: 15


,id,title,year,type,country,genres,synopsis,tags,cast,director,rating,votes,episodes,platform,source
0,1,!Women Art Revolution,2010.0,Movie,NaN,Documentary,NaN,NaN,NaN,NaN,4.3,2.0,NaN,NaN,demographic
1,2,"""Sherlock"" The Great Game",2010.0,Movie,Great Britain,Thriller,Mycroft would like his brother Sherlock to inv...,NaN,"Martin Freeman, Benedict Cumberbatch, Rupert G...",Paul McGuigan,7.5,39.0,NaN,NaN,filmtv


# Data Understanding

In [10]:
print(df.dtypes)
print()
print(df.isna().sum())
print()
print(df["type"].value_counts())

id                   int64
title               object
year                 int32
type                object
country             object
genres              object
synopsis            object
tags                object
cast                object
director            object
rating             float64
votes              float64
episodes           float64
platform            object
source              object
genres_list         object
cast_list           object
weighted_rating    float64
dtype: object

id                     0
title                  0
year                   0
type                   0
country                0
genres                 0
synopsis               0
tags                   0
cast                   0
director               0
rating                 0
votes                  0
episodes           75123
platform           69618
source                 0
genres_list            0
cast_list              0
weighted_rating        0
dtype: int64

type
Movie       71674
Tv/Drama    

In [11]:
def parse_list(nilai, batas=None):
    if isinstance(nilai, list):
        hasil = nilai
    elif isinstance(nilai, str) and nilai.strip():
        teks = nilai.strip()
        if teks.startswith("["):
            try:
                hasil = ast.literal_eval(teks)
            except (ValueError, SyntaxError):
                hasil = teks.strip("[]").split(",")
        else:
            hasil = teks.split(",")
    else:
        hasil = []
    hasil = [str(x).strip().strip("'\"") for x in hasil if str(x).strip()]
    return hasil[:batas] if batas else hasil

# Pembersihan Data

In [12]:
data = df.copy()

data["year"] = pd.to_numeric(data["year"], errors="coerce")
data["rating"] = pd.to_numeric(data["rating"], errors="coerce").fillna(0.0)
data["votes"] = pd.to_numeric(data["votes"], errors="coerce").fillna(0).astype(int)
data = data.dropna(subset=["title", "year"]).copy()
data["year"] = data["year"].astype(int)

for kolom in ["type", "country"]:
    data[kolom] = data[kolom].fillna("Unknown")
for kolom in ["synopsis", "tags"]:
    data[kolom] = data[kolom].fillna("")

data["genres"] = data["genres"].apply(parse_list)
data["cast"] = data["cast"].apply(lambda x: parse_list(x, 5))
data["directors"] = data["director"].apply(parse_list)
data["teks"] = (data["synopsis"] + " " + data["tags"]).str.strip()

data = data.sort_values("votes", ascending=False)
data = data.drop_duplicates(subset=["title", "year", "type"], keep="first")
data = data.reset_index(drop=True)
data["label"] = data["title"] + " (" + data["year"].astype(str) + ")"

kolom_simpan = ["id", "title", "label", "year", "type", "country", "genres", "cast",
                "directors", "synopsis", "teks", "rating", "votes", "episodes", "platform"]
data = data[kolom_simpan]

print(f"Sebelum: {len(df)} baris | Sesudah dibersihkan: {len(data)} baris")
data.head(3)

Sebelum: 82830 baris | Sesudah dibersihkan: 82830 baris


,id,title,label,year,type,country,genres,cast,directors,synopsis,teks,rating,votes,episodes,platform
0,72204,The Shawshank Redemption,The Shawshank Redemption (1994),1994,Movie,United States,[Drama],"[Tim Robbins, Morgan Freeman, James Whitmore, ...",[Frank Darabont],We are in 1946. Tried for the murder of his wi...,We are in 1946. Tried for the murder of his wi...,9.3,2474122,NaN,NaN
1,65405,The Dark Knight,The Dark Knight (2008),2008,Movie,United States,"[Action, Crime, Drama]","[Christian Bale, Heath Ledger, Morgan Freeman,...",[Christopher Nolan],Batman and Commissioner Gordon team up with Ha...,Batman and Commissioner Gordon team up with Ha...,9.0,2428117,NaN,NaN
2,31180,Inception,Inception (2010),2010,Movie,"United States, United Kingdom","[Action, Adventure, Science Fiction, Fantasy, ...","[Leonardo DiCaprio, Joseph Gordon Levitt, Elli...",[Christopher Nolan],A troubled thief who extracts secrets from peo...,A troubled thief who extracts secrets from peo...,8.8,2178860,NaN,Netflix


# Hitung Rating

In [13]:
M_QUANTILE = 0.80

C = data.loc[data["votes"] > 0, "rating"].mean()
m = max(data["votes"].quantile(M_QUANTILE), 1)

v = data["votes"]
R = data["rating"]
data["wr"] = (v / (v + m)) * R + (m / (v + m)) * C

print(f"C (rata-rata rating) = {C:.2f}")
print(f"m (ambang voter)     = {m:.0f}")

C (rata-rata rating) = 6.08
m (ambang voter)     = 83


In [14]:
kolom = ["title", "year", "rating", "votes", "wr"]

print("Top 10 berdasarkan rating mentah:")
display(data.sort_values("rating", ascending=False)[kolom].head(10))

print("Top 10 berdasarkan weighted rating:")
display(data.sort_values("wr", ascending=False)[kolom].head(10))

Top 10 berdasarkan rating mentah:


,title,year,rating,votes,wr
72430,"My Night, Your Day",2015,10.0,1,6.124449
73686,Shark Kill,1976,10.0,1,6.124449
70625,Sebastian Maniscalco: Why Would You Do That,2016,10.0,2,6.170044
64091,Butterfly,2015,10.0,3,6.214578
73741,Forever,2006,10.0,1,6.124449
70584,Siccin 3: Cürmü Ask,2016,10.0,2,6.170044
73758,Dragonfly,1976,10.0,1,6.124449
70522,Katt Williams: Priceless: Afterlife,2014,10.0,2,6.170044
73796,Road to Redemption,2001,10.0,1,6.124449
73802,Firefly,1975,10.0,1,6.124449


Top 10 berdasarkan weighted rating:


,title,year,rating,votes,wr
639,Aspirants,2004,9.7,286206,9.698950
3749,The Chosen,2007,9.6,21759,9.586615
5683,Bluey,2005,9.6,4110,9.530278
1528,Scam 1992: The Harshad Mehta Story,2020,9.5,131914,9.497848
7820,Run BTS!,2015,9.7,1013,9.425688
12,Breaking Bad,2021,9.4,1593305,9.399827
3147,When Life Gives You Tangerines,2025,9.4,36925,9.392549
8016,BTS: Bon Voyage,2016,9.6,908,9.304999
0,The Shawshank Redemption,1994,9.3,2474122,9.299892
569,The Wire,2021,9.3,306334,9.299127


# TF-IDF

In [15]:
def bersihkan(daftar):
    return [x.lower().replace(" ", "") for x in daftar if x]


def buat_soup(genres, cast, directors, country):
    negara = [] if country == "Unknown" else [country]
    token = (
        bersihkan(genres) * 2
        + bersihkan(cast)
        + bersihkan(directors) * 3
        + bersihkan(negara)
    )
    return " ".join(token)


soup = [
    buat_soup(g, c, d, n)
    for g, c, d, n in zip(data["genres"], data["cast"], data["directors"], data["country"])
]

tfidf_meta = TfidfVectorizer(token_pattern=r"\S+")
meta_matrix = tfidf_meta.fit_transform(soup)

tfidf_teks = TfidfVectorizer(
    stop_words="english", ngram_range=(1, 2), min_df=3, max_features=30000
)
teks_matrix = tfidf_teks.fit_transform(data["teks"])

print("meta_matrix:", meta_matrix.shape)
print("teks_matrix:", teks_matrix.shape)

meta_matrix: (82830, 98413)
teks_matrix: (82830, 30000)


# Simpan di Pickel

In [18]:
paket = {
    "data": data,
    "meta_matrix": meta_matrix,
    "teks_matrix": teks_matrix,
    "C": float(C),
    "m": float(m),
}

with open("movie_marger.pkl", "wb") as f:
    pickle.dump(paket, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Tersimpan:", round(os.path.getsize("movie_marger.pkl") / 1024 / 1024, 1), "MB")

Tersimpan: 118.3 MB


# Engine

In [37]:
%%writefile engine.py
import pickle
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process, utils
from sklearn.metrics.pairwise import cosine_similarity


@dataclass
class Filters:
    year_range: Optional[tuple] = None
    types: list = field(default_factory=list)
    genres: list = field(default_factory=list)
    genre_mode: str = "any"
    min_rating: float = 0.0
    min_votes: int = 0
    countries: list = field(default_factory=list)


def muat_paket(path):
    with open(path, "rb") as f:
        paket = pickle.load(f)
    paket["titles"] = paket["data"]["title"].tolist()
    paket["titles_proc"] = [utils.default_process(t) for t in paket["titles"]]
    paket["genre_sets"] = [set(g) for g in paket["data"]["genres"]]
    return paket


def minmax(x):
    rentang = x.max() - x.min()
    if rentang == 0:
        return np.zeros_like(x, dtype=float)
    return (x - x.min()) / rentang


def cari_judul(paket, query, limit=5, score_cutoff=50):
    query = (query or "").strip()
    if not query:
        return []
    data = paket["data"]
    hasil_mentah = process.extract(
        utils.default_process(query),
        paket["titles_proc"],
        scorer=fuzz.WRatio,
        limit=limit * 3,
        score_cutoff=score_cutoff,
    )
    hasil = []
    sudah = set()
    for _, skor, idx in hasil_mentah:
        if idx in sudah:
            continue
        sudah.add(idx)
        baris = data.iloc[idx]
        hasil.append({
            "idx": int(idx),
            "title": baris["title"],
            "year": int(baris["year"]),
            "label": baris["label"],
            "score": float(skor),
        })
        if len(hasil) >= limit:
            break
    return hasil


def buat_mask(paket, f):
    data = paket["data"]
    mask = np.ones(len(data), dtype=bool)
    if f is None:
        return mask

    if f.year_range:
        mask &= data["year"].between(*f.year_range).to_numpy()

    if f.types:
        mask &= data["type"].isin(f.types).to_numpy()

    if f.genres:
        agg = all if f.genre_mode == "all" else any
        mask &= np.array([agg(g in s for g in f.genres) for s in paket["genre_sets"]])

    if f.min_rating:
        mask &= (data["rating"] >= f.min_rating).to_numpy()

    if f.min_votes:
        mask &= (data["votes"] >= f.min_votes).to_numpy()

    if f.countries:
        mask &= data["country"].isin(f.countries).to_numpy()

    return mask


def jelaskan(paket, asal, tujuan, sim_teks):
    alasan = []

    genre_sama = [g for g in asal["genres"] if g in tujuan["genres"]]
    if genre_sama:
        alasan.append("Genre sama: " + ", ".join(genre_sama))

    sutradara_sama = [d for d in asal["directors"] if d in tujuan["directors"]]
    if sutradara_sama:
        alasan.append("Sutradara sama: " + ", ".join(sutradara_sama))

    pemeran_sama = [c for c in asal["cast"] if c in tujuan["cast"]]
    if pemeran_sama:
        alasan.append("Pemeran sama: " + ", ".join(pemeran_sama[:3]))

    if sim_teks >= 0.15:
        alasan.append("Sinopsis bernuansa mirip")

    if asal["country"] == tujuan["country"] and tujuan["country"] != "Unknown":
        alasan.append(f"Sama-sama produksi {tujuan['country']}")

    if tujuan["votes"] >= paket["m"] and tujuan["rating"] >= 7.0:
        alasan.append(
            f"Berkualitas: rating {tujuan['rating']:.1f} dari {tujuan['votes']:,} voter"
        )

    return alasan or ["Profil konten secara keseluruhan mirip"]


def rekomendasi(paket, idx, top_n=10, alpha=0.7, w_meta=0.7, filters=None, pool=150):
    data = paket["data"]

    sim_meta = cosine_similarity(paket["meta_matrix"][idx], paket["meta_matrix"]).ravel()
    sim_teks = cosine_similarity(paket["teks_matrix"][idx], paket["teks_matrix"]).ravel()
    sim = w_meta * sim_meta + (1 - w_meta) * sim_teks

    mask = buat_mask(paket, filters)
    mask[idx] = False
    kandidat = np.where(mask)[0]
    if len(kandidat) == 0:
        return pd.DataFrame()

    kandidat = kandidat[np.argsort(sim[kandidat])[::-1][:pool]]
    sim_n = minmax(sim[kandidat])
    wr_n = minmax(data["wr"].to_numpy()[kandidat])
    skor = alpha * sim_n + (1 - alpha) * wr_n

    urutan = np.argsort(skor)[::-1][:top_n]
    asal = data.iloc[idx]
    baris_hasil = []
    for o in urutan:
        j = int(kandidat[o])
        tujuan = data.iloc[j]
        alasan = jelaskan(paket, asal, tujuan, sim_teks[j])
        baris_hasil.append({
            "idx": j,
            "title": tujuan["title"],
            "year": int(tujuan["year"]),
            "type": tujuan["type"],
            "country": tujuan["country"],
            "genres": ", ".join(tujuan["genres"]),
            "director": ", ".join(tujuan["directors"]),
            "episodes": tujuan["episodes"],
            "platform": tujuan["platform"],
            "rating": float(tujuan["rating"]),
            "votes": int(tujuan["votes"]),
            "weighted_rating": round(float(tujuan["wr"]), 2),
            "similarity": round(float(sim[j]), 3),
            "score": round(float(skor[o]), 3),
            "reasons": alasan,
            "explanation": " · ".join(alasan),
            "synopsis": tujuan["synopsis"],
        })
    return pd.DataFrame(baris_hasil)


def terbaik(paket, filters=None, top_n=20):
    data = paket["data"]
    mask = buat_mask(paket, filters)
    d = data[mask].sort_values("wr", ascending=False).head(top_n)
    out = d[["title", "year", "type", "country", "genres", "rating", "votes", "wr"]].copy()
    out["genres"] = out["genres"].apply(", ".join)
    out = out.rename(columns={"wr": "weighted_rating"})
    return out.round({"weighted_rating": 2}).reset_index(drop=True)

Overwriting engine.py


In [38]:
import importlib
import engine

importlib.reload(engine)

from engine import Filters, muat_paket, cari_judul, rekomendasi, terbaik

paket = muat_paket("movie_marger.pkl")
print(len(paket["data"]), "judul siap dipakai")

82830 judul siap dipakai


## Demo Pencarian Typo

In [39]:
for q in ["crash landin on you", "gobllin", "decendants of the son", "avangers"]:
    hasil = cari_judul(paket, q, limit=3)
    print(f"{q!r:28} ->", [(h["label"], round(h["score"])) for h in hasil])

'crash landin on you'        -> [('Crash Landing on You (2019)', 97), ('Crash (2004)', 90), ('You (1984)', 90)]
'gobllin'                    -> [('Goblin (2016)', 92), ('Goblin (2010)', 92), ('Go (1999)', 90)]
'decendants of the son'      -> [('Descendants of the Sun (2016)', 93), ('Descendant of the Sun (1983)', 90), ('Son (2021)', 90)]
'avangers'                   -> [('ER (1999)', 90), ('Ava (2020)', 90), ('An (2015)', 90)]


## Tampil rekomendasi dan penjelasannya

In [40]:
def tampil(judul, **kw):
    cocok = cari_judul(paket, judul, limit=1)[0]
    print(f"Karena kamu suka: {cocok['label']} (kecocokan {cocok['score']:.0f}%)\n")
    recs = rekomendasi(paket, cocok["idx"], **kw)
    for i, r in recs.iterrows():
        print(f"{i + 1}. {r.title} ({r.year}) | skor {r.score:.2f} | ⭐ {r.rating} dari {r.votes:,} voter")
        print(f"   ↳ {r.explanation}\n")
    return recs


recs = tampil("crash landin on you", top_n=8)

Karena kamu suka: Crash Landing on You (2019) (kecocokan 97%)

1. The Good Wife (2016) | skor 0.91 | ⭐ 8.0 dari 2,118 voter
   ↳ Genre sama: Romance, Political · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi South Korea · Berkualitas: rating 8.0 dari 2,118 voter

2. Doona! (2023) | skor 0.83 | ⭐ 7.7 dari 35,619 voter
   ↳ Genre sama: Romance · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi South Korea · Berkualitas: rating 7.7 dari 35,619 voter

3. Cruel City (2013) | skor 0.82 | ⭐ 8.5 dari 10,227 voter
   ↳ Genre sama: Romance · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi South Korea · Berkualitas: rating 8.5 dari 10,227 voter

4. Witch's Romance (2014) | skor 0.81 | ⭐ 7.9 dari 13,912 voter
   ↳ Genre sama: Comedy, Romance · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi South Korea · Berkualitas: rating 7.9 dari 13,912 voter

5. Life on Mars (2018) | skor 0.76 | ⭐ 8.7 dari 12,123 voter
   ↳ Genre sama: Comedy · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi So

## Pengaruh Alpha (kemiripan vs kualitas)

In [41]:
idx_film = cari_judul(paket, "crash landin on you", limit=1)[0]["idx"]

for a in (1.0, 0.7, 0.3):
    r = rekomendasi(paket, idx_film, top_n=5, alpha=a)
    print(f"alpha = {a}")
    display(r[["title", "year", "similarity", "weighted_rating", "score"]])

alpha = 1.0


,title,year,similarity,weighted_rating,score
0,The Good Wife,2016,0.479,7.93,1.000
1,Doona!,2023,0.449,7.70,0.912
2,Witch's Romance,2014,0.435,7.89,0.872
3,Cruel City,2013,0.414,8.48,0.809
4,Because It's the First Time,2015,0.401,7.48,0.771


alpha = 0.7


,title,year,similarity,weighted_rating,score
0,The Good Wife,2016,0.479,7.93,0.906
1,Doona!,2023,0.449,7.70,0.826
2,Cruel City,2013,0.414,8.48,0.818
3,Witch's Romance,2014,0.435,7.89,0.814
4,Life on Mars,2018,0.379,8.68,0.764


alpha = 0.3


,title,year,similarity,weighted_rating,score
0,Life on Mars,2018,0.379,8.68,0.838
1,Cruel City,2013,0.414,8.48,0.830
2,The Good Wife,2016,0.479,7.93,0.781
3,Romance Is a Bonus Book,2019,0.396,8.10,0.741
4,Witch's Romance,2014,0.435,7.89,0.736


## rekomendasi dengan filter

In [42]:
f = Filters(
    year_range=(2015, 2025),
    types=[],
    genres=["Romance", "Comedy"],
    genre_mode="any",
    min_rating=7.0,
    min_votes=100,
    countries=[],
)

recs = tampil("crash landin on you", top_n=8, filters=f)

Karena kamu suka: Crash Landing on You (2019) (kecocokan 97%)

1. The Good Wife (2016) | skor 0.86 | ⭐ 8.0 dari 2,118 voter
   ↳ Genre sama: Romance, Political · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi South Korea · Berkualitas: rating 8.0 dari 2,118 voter

2. Life on Mars (2018) | skor 0.79 | ⭐ 8.7 dari 12,123 voter
   ↳ Genre sama: Comedy · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi South Korea · Berkualitas: rating 8.7 dari 12,123 voter

3. Doona! (2023) | skor 0.78 | ⭐ 7.7 dari 35,619 voter
   ↳ Genre sama: Romance · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi South Korea · Berkualitas: rating 7.7 dari 35,619 voter

4. Romance Is a Bonus Book (2019) | skor 0.74 | ⭐ 8.1 dari 64,520 voter
   ↳ Genre sama: Comedy, Romance · Sutradara sama: Lee Jeong Hyo · Sama-sama produksi South Korea · Berkualitas: rating 8.1 dari 64,520 voter

5. Because It's the First Time (2015) | skor 0.67 | ⭐ 7.5 dari 5,565 voter
   ↳ Genre sama: Romance · Sutradara sama: Lee Jeong Hyo

## judul terbaik berdasarkan weighted rating

In [43]:
terbaik(paket, Filters(year_range=(2010, 2025), genres=["Drama"]), top_n=10)

,title,year,type,country,genres,rating,votes,weighted_rating
0,Scam 1992: The Harshad Mehta Story,2020,Tv/Drama,Unknown,"Biography, Crime, Drama",9.5,131914,9.50
1,Breaking Bad,2021,Tv/Drama,Unknown,"Crime, Drama, Thriller",9.4,1593305,9.40
2,When Life Gives You Tangerines,2025,Tv/Drama,South Korea,"Romance, Life, Drama",9.4,36925,9.39
3,The Wire,2021,Tv/Drama,Unknown,"Crime, Drama, Thriller",9.3,306334,9.30
4,Toma,2021,Movie,Unknown,"Biography, Drama, Music",9.3,5562,9.25
5,Game of Thrones,2021,Tv/Drama,Unknown,"Action, Adventure, Drama",9.2,1885776,9.20
6,The Sopranos,2021,Tv/Drama,Unknown,"Crime, Drama",9.2,340006,9.20
7,Twinkling Watermelon,2023,Tv/Drama,South Korea,"Romance, Youth, Drama, Fantasy",9.2,74669,9.20
8,Sardar Udham,2021,Movie,Unknown,"Biography, Crime, Drama",9.2,8649,9.17
9,Leyla ile Mecnun,2010,Tv/Drama,Unknown,"Adventure, Comedy, Drama",9.1,91862,9.10
